# Silver — Qualidade e Tratamento dos Dados

Este notebook contempla as transformações de qualidade, tipagem e padronização aplicadas às tabelas da camada Bronze para preparação dos dados utilizados na construção do modelo analítico.

As regras implementadas nesta etapa foram definidas a partir do profiling e das análises exploratórias realizadas anteriormente na camada Bronze.

Para cada conjunto de dados são apresentados os problemas ou características identificadas, a regra de tratamento adotada e a validação do resultado após a transformação.

## 01. Preparação da Camada Silver

A camada Silver é utilizada para armazenar os dados após a aplicação das regras de qualidade, tipagem e padronização. As tabelas permanecem próximas à granularidade das fontes de origem, mas são preparadas para utilização na construção do modelo analítico da camada Gold.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

## 02. Tratamento de Pedidos


### 02.1 Análise de Qualidade

Nesta etapa são avaliadas as características da tabela `bronze.orders` que podem demandar tratamento antes de sua persistência na camada Silver, considerando tipos de dados, valores nulos, consistência dos registros e regras relacionadas ao ciclo dos pedidos.

In [0]:
%sql
-- Analise para entender os status em que os NULLS aparecem
SELECT
    order_status,
    COUNT(*) AS qtd_pedidos,

    SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) 
        AS nulos_aprovacao,

    SUM(CASE WHEN order_delivered_carrier_date IS NULL THEN 1 ELSE 0 END) 
        AS nulos_envio_transportadora,

    SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) 
        AS nulos_entrega_cliente,

    SUM(CASE WHEN order_estimated_delivery_date IS NULL THEN 1 ELSE 0 END) 
        AS nulos_previsao_entrega

FROM workspace.bronze.orders

GROUP BY order_status
ORDER BY qtd_pedidos DESC;

order_status,qtd_pedidos,nulos_aprovacao,nulos_envio_transportadora,nulos_entrega_cliente,nulos_previsao_entrega
delivered,96478,14,2,8,0
shipped,1107,0,0,1107,0
canceled,625,141,550,619,0
unavailable,609,0,609,609,0
invoiced,314,0,314,314,0
processing,301,0,301,301,0
created,5,5,5,5,0
approved,2,0,2,2,0


In [0]:
%sql
-- Validacao se os campos de data como STRING realmente sao datas para ajuste de tipagem
SELECT
    SUM(
        CASE 
            WHEN order_purchase_timestamp IS NOT NULL
             AND TRY_CAST(order_purchase_timestamp AS TIMESTAMP) IS NULL
            THEN 1 ELSE 0 
        END
    ) AS compra_invalida,

    SUM(
        CASE 
            WHEN order_approved_at IS NOT NULL
             AND TRY_CAST(order_approved_at AS TIMESTAMP) IS NULL
            THEN 1 ELSE 0 
        END
    ) AS aprovacao_invalida,

    SUM(
        CASE 
            WHEN order_delivered_carrier_date IS NOT NULL
             AND TRY_CAST(order_delivered_carrier_date AS TIMESTAMP) IS NULL
            THEN 1 ELSE 0 
        END
    ) AS envio_invalido,

    SUM(
        CASE 
            WHEN order_delivered_customer_date IS NOT NULL
             AND TRY_CAST(order_delivered_customer_date AS TIMESTAMP) IS NULL
            THEN 1 ELSE 0 
        END
    ) AS entrega_invalida,

    SUM(
        CASE 
            WHEN order_estimated_delivery_date IS NOT NULL
             AND TRY_CAST(order_estimated_delivery_date AS TIMESTAMP) IS NULL
            THEN 1 ELSE 0 
        END
    ) AS previsao_invalida

FROM workspace.bronze.orders;

compra_invalida,aprovacao_invalida,envio_invalido,entrega_invalida,previsao_invalida
0,0,0,0,0


### 02.2 Regras e Transformações


A análise identificou que os campos relacionados às datas do ciclo do pedido estão armazenados como `STRING`, embora todos os valores não nulos apresentem formato válido para conversão.

Também foram identificados valores nulos em datas de aprovação, envio e entrega. A maior parte dessas ausências é compatível com o status do pedido e representa etapas do ciclo que ainda não ocorreram ou não foram concluídas.

Foram identificados ainda poucos pedidos com status `delivered` que apresentam ausência em alguma data intermediária ou de entrega. Como não existem informações suficientes na fonte para reconstruir essas datas com segurança, os valores serão preservados como nulos, evitando a imputação de informações não observadas.

Dessa forma, foram definidas as seguintes regras:

- conversão dos campos de data e hora de `STRING` para `TIMESTAMP`;
- manutenção dos valores nulos existentes nas datas do ciclo do pedido;
- preservação da granularidade original de uma linha por pedido;
- manutenção dos demais campos sem alteração de conteúdo.

In [0]:
%sql
-- Converte os campos temporais de pedidos para TIMESTAMP preservando os demais valores da fonte.

CREATE OR REPLACE TABLE workspace.silver.orders
USING DELTA
AS
SELECT
    order_id,
    customer_id,
    order_status,
    CAST(order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
    CAST(order_approved_at AS TIMESTAMP) AS order_approved_at,
    CAST(order_delivered_carrier_date AS TIMESTAMP) AS order_delivered_carrier_date,
    CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_date,
    CAST(order_estimated_delivery_date AS TIMESTAMP) AS order_estimated_delivery_date
FROM workspace.bronze.orders;

num_affected_rows,num_inserted_rows


### 02.3 Validação

Após a persistência, são realizadas validações para verificar a manutenção da granularidade, a quantidade de registros e a aplicação correta das transformações definidas.

In [0]:
%sql
-- Valida a quantidade de registros e a unicidade de order_id após o tratamento.

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT order_id) AS qtd_pedidos,
    COUNT(*) - COUNT(DISTINCT order_id) AS qtd_duplicados
FROM workspace.silver.orders;

qtd_registros,qtd_pedidos,qtd_duplicados
99441,99441,0


In [0]:
%sql
DESCRIBE workspace.silver.orders;

col_name,data_type,comment
order_id,string,Identificador único do pedido. Domínio: identificador alfanumérico único e não nulo.
customer_id,string,Identificador do cliente associado ao pedido. Domínio: identificador alfanumérico não nulo.
order_status,string,"Status do pedido. Domínio observado: delivered, shipped, canceled, unavailable, invoiced, processing, created e approved. Não apresenta valores nulos."
order_purchase_timestamp,timestamp,null
order_approved_at,timestamp,null
order_delivered_carrier_date,timestamp,null
order_delivered_customer_date,timestamp,null
order_estimated_delivery_date,timestamp,null


## 03. Tratamento de Itens de Pedido


### 03.1 Análise de Qualidade

A tabela `bronze.order_items` contém os itens associados aos pedidos, com granularidade de uma linha por combinação de pedido e item.

Nesta etapa são avaliadas características relacionadas à integridade das chaves, tipos de dados e valores das métricas de preço e frete, com o objetivo de preparar os dados para utilização na camada analítica.

In [0]:
%sql
-- Valida se todos os valores de shipping_limit_date podem ser convertidos para TIMESTAMP.

SELECT
    SUM(
        CASE
            WHEN shipping_limit_date IS NOT NULL
             AND TRY_CAST(shipping_limit_date AS TIMESTAMP) IS NULL
            THEN 1 ELSE 0
        END
    ) AS datas_invalidas
FROM workspace.bronze.order_items;

datas_invalidas
0


In [0]:
%sql
-- Identifica itens com limite de envio posterior a 2018 para avaliar possíveis valores atípicos.

SELECT
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value
FROM workspace.bronze.order_items
WHERE TRY_CAST(shipping_limit_date AS TIMESTAMP) >= TIMESTAMP '2019-01-01 00:00:00'
ORDER BY TRY_CAST(shipping_limit_date AS TIMESTAMP);

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
9c94a4ea2f7876660fa6f1b59b69c8e6,1,282b126b2354516c5f400154398f616d,7a241947449cc45dbfda4f9d0798d9d0,2020-02-03 20:23:22,75.99,14.7
13bdf405f961a6deec817d817f5c6624,1,96ea060e41bdecc64e2de00b97068975,7a241947449cc45dbfda4f9d0798d9d0,2020-02-05 03:30:51,69.99,14.66
c2bb89b5c1dd978d507284be78a04cb2,1,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,99.99,61.44
c2bb89b5c1dd978d507284be78a04cb2,2,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,99.99,61.44


In [0]:
%sql
-- Compara datas de limite de envio atípicas com a data de compra do respectivo pedido.

SELECT
    i.order_id,
    i.order_item_id,
    o.order_status,
    o.order_purchase_timestamp,
    i.shipping_limit_date,
    DATEDIFF(
        TRY_CAST(i.shipping_limit_date AS DATE),
        TRY_CAST(o.order_purchase_timestamp AS DATE)
    ) AS dias_entre_compra_limite_envio
FROM workspace.bronze.order_items i
LEFT JOIN workspace.bronze.orders o
    ON i.order_id = o.order_id
WHERE TRY_CAST(i.shipping_limit_date AS TIMESTAMP) >= TIMESTAMP '2019-01-01 00:00:00'
ORDER BY TRY_CAST(i.shipping_limit_date AS TIMESTAMP);

order_id,order_item_id,order_status,order_purchase_timestamp,shipping_limit_date,dias_entre_compra_limite_envio
9c94a4ea2f7876660fa6f1b59b69c8e6,1,shipped,2017-03-14 19:23:22,2020-02-03 20:23:22,1056
13bdf405f961a6deec817d817f5c6624,1,canceled,2017-03-16 02:30:51,2020-02-05 03:30:51,1056
c2bb89b5c1dd978d507284be78a04cb2,1,delivered,2017-05-23 22:28:36,2020-04-09 22:35:08,1052
c2bb89b5c1dd978d507284be78a04cb2,2,delivered,2017-05-23 22:28:36,2020-04-09 22:35:08,1052


### 03.2 Regras e Transformações

A análise identificou que todos os valores de shipping_limit_date possuem formato válido para conversão. Foram encontrados quatro itens, associados a três pedidos, com datas de limite de envio em 2020, aproximadamente três anos após a realização da compra.

Esses registros foram considerados valores atípicos da fonte. Como não existem informações suficientes para determinar qual seria a data correta, os valores foram preservados, evitando a aplicação de correções ou imputações sem evidência.

Também foi identificada a oportunidade de adequar os campos monetários, originalmente armazenados como DOUBLE, para um tipo decimal de precisão fixa.

Foram definidas as seguintes regras:

- conversão de shipping_limit_date de STRING para TIMESTAMP;
- conversão de price e freight_value de DOUBLE para DECIMAL(10,2);
- preservação dos valores atípicos de shipping_limit_date, mantendo fidelidade à fonte;
- preservação da granularidade de uma linha por combinação de pedido e item;
- manutenção dos demais campos sem alteração de conteúdo.

In [0]:
%sql
-- Adequa os tipos de data e valores monetários dos itens de pedido.

CREATE OR REPLACE TABLE workspace.silver.order_items
USING DELTA
AS
SELECT
    order_id,
    order_item_id,
    product_id,
    seller_id,
    CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_date,
    CAST(price AS DECIMAL(10,2)) AS price,
    CAST(freight_value AS DECIMAL(10,2)) AS freight_value
FROM workspace.bronze.order_items;

num_affected_rows,num_inserted_rows


### 03.3 Validação

Após a persistência, são realizadas validações para verificar a manutenção da granularidade, a quantidade de registros e a aplicação correta das transformações definidas.

In [0]:
%sql
-- Valida a quantidade de registros e a unicidade da combinação order_id + order_item_id.

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT CONCAT(order_id, '|', order_item_id)) AS qtd_itens_unicos,
    COUNT(*) - COUNT(DISTINCT CONCAT(order_id, '|', order_item_id)) AS qtd_duplicados
FROM workspace.silver.order_items;

qtd_registros,qtd_itens_unicos,qtd_duplicados
112650,112650,0


In [0]:
%sql
-- Valida se a conversão para DECIMAL(10,2) preservou os valores de preço e frete.

SELECT
    SUM(
        CASE 
            WHEN CAST(b.price AS DECIMAL(10,2)) <> s.price
            THEN 1 ELSE 0 
        END
    ) AS divergencias_price,

    SUM(
        CASE 
            WHEN CAST(b.freight_value AS DECIMAL(10,2)) <> s.freight_value
            THEN 1 ELSE 0 
        END
    ) AS divergencias_frete

FROM workspace.bronze.order_items b
INNER JOIN workspace.silver.order_items s
    ON b.order_id = s.order_id
   AND b.order_item_id = s.order_item_id;

divergencias_price,divergencias_frete
0,0


In [0]:
%sql
-- Valida os tipos de dados resultantes após as transformações.

DESCRIBE workspace.silver.order_items;

col_name,data_type,comment
order_id,string,Identificador do pedido ao qual o item pertence. Domínio: identificador alfanumérico não nulo. Foram observados 98.666 pedidos distintos.
order_item_id,bigint,"Identificador sequencial do item dentro do pedido. Domínio observado: valores inteiros de 1 a 21, sem valores nulos. Em conjunto com order_id identifica unicamente um item do pedido."
product_id,string,Identificador do produto associado ao item do pedido. Domínio: identificador alfanumérico não nulo. Foram observados 32.951 produtos distintos.
seller_id,string,Identificador do vendedor responsável pelo item. Domínio: identificador alfanumérico não nulo. Foram observados 3.095 vendedores distintos.
shipping_limit_date,timestamp,null
price,"decimal(10,2)",null
freight_value,"decimal(10,2)",null


## 04. Tratamento de Produtos e Categorias

### 04.1 Análise de Qualidade

As tabelas `bronze.products` e `bronze.category_translation` contêm, respectivamente, as características dos produtos e a tradução das categorias de produtos para o inglês.

Nesta etapa são avaliados valores ausentes nos atributos dos produtos e a cobertura da tabela de tradução de categorias, com o objetivo de preparar os dados para posterior construção da dimensão de produtos na camada Gold.

In [0]:
%sql
-- Verifica o padrão de ausência dos principais atributos descritivos dos produtos.

SELECT
    COUNT(*) AS qtd_produtos,
    SUM(CASE WHEN product_category_name IS NULL THEN 1 ELSE 0 END) AS sem_categoria,
    SUM(CASE WHEN product_name_lenght IS NULL THEN 1 ELSE 0 END) AS sem_tamanho_nome,
    SUM(CASE WHEN product_description_lenght IS NULL THEN 1 ELSE 0 END) AS sem_descricao,
    SUM(CASE WHEN product_photos_qty IS NULL THEN 1 ELSE 0 END) AS sem_fotos,
    SUM(
        CASE
            WHEN product_category_name IS NULL
             AND product_name_lenght IS NULL
             AND product_description_lenght IS NULL
             AND product_photos_qty IS NULL
            THEN 1 ELSE 0
        END
    ) AS nulos_conjuntos
FROM workspace.bronze.products;

qtd_produtos,sem_categoria,sem_tamanho_nome,sem_descricao,sem_fotos,nulos_conjuntos
32951,610,610,610,610,610


In [0]:
%sql
-- Identifica produtos com ausência em atributos físicos para avaliar o padrão dos valores nulos.

SELECT
    product_id,
    product_category_name,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
FROM workspace.bronze.products
WHERE product_weight_g IS NULL
   OR product_length_cm IS NULL
   OR product_height_cm IS NULL
   OR product_width_cm IS NULL;

product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
5eb564652db742ff8f28759cd8d2652a,null,null,null,null,null
09ff539a621711667c43eba6a3bd8466,bebes,null,null,null,null


In [0]:
%sql
-- Identifica categorias de produtos sem correspondência na tabela de tradução.

SELECT
    p.product_category_name,
    COUNT(*) AS qtd_produtos
FROM workspace.bronze.products p
LEFT JOIN workspace.bronze.category_translation t
    ON p.product_category_name = t.product_category_name
WHERE p.product_category_name IS NOT NULL
  AND t.product_category_name IS NULL
GROUP BY p.product_category_name
ORDER BY qtd_produtos DESC;

product_category_name,qtd_produtos
portateis_cozinha_e_preparadores_de_alimentos,10
pc_gamer,3


In [0]:
%sql
-- Verifica se os atributos de contagem possuem somente valores inteiros antes da conversão de tipo.

SELECT
    SUM(CASE 
        WHEN product_name_lenght IS NOT NULL
         AND product_name_lenght <> FLOOR(product_name_lenght)
        THEN 1 ELSE 0 
    END) AS nome_nao_inteiro,

    SUM(CASE 
        WHEN product_description_lenght IS NOT NULL
         AND product_description_lenght <> FLOOR(product_description_lenght)
        THEN 1 ELSE 0 
    END) AS descricao_nao_inteira,

    SUM(CASE 
        WHEN product_photos_qty IS NOT NULL
         AND product_photos_qty <> FLOOR(product_photos_qty)
        THEN 1 ELSE 0 
    END) AS fotos_nao_inteiras

FROM workspace.bronze.products;

nome_nao_inteiro,descricao_nao_inteira,fotos_nao_inteiras
0,0,0


### 04.2 Regras e Transformações

A análise identificou 610 produtos com ausência simultânea de categoria, tamanho do nome, tamanho da descrição e quantidade de fotos. Também foram identificados dois produtos sem informações de peso e dimensões físicas.

Como não existem informações suficientes na fonte para estimar os atributos quantitativos ausentes, esses valores serão preservados como nulos, evitando imputações sem evidência.

Para o atributo de categoria, a ausência será padronizada como sem_categoria, permitindo que esses produtos permaneçam identificáveis nas análises por categoria.

Também foram identificadas duas categorias presentes na tabela de produtos sem correspondência na tabela de tradução, totalizando 13 produtos. Nesses casos, o nome original da categoria será preservado como alternativa à tradução, evitando a geração de categorias nulas no modelo analítico.

Os campos relacionados ao tamanho do nome, tamanho da descrição e quantidade de fotos foram originalmente armazenados como DOUBLE. Após a validação de que seus valores representam contagens inteiras, esses campos serão convertidos para INT, adequando o tipo de dado ao significado das informações.

Foram definidas as seguintes regras:

- substituição de categorias nulas por sem_categoria;
- preservação dos atributos quantitativos ausentes como nulos;
- utilização do nome original da categoria quando não houver tradução disponível;
- conversão de product_name_length, product_description_length e product_photos_qty para INT;
- preservação da granularidade de uma linha por produto;

In [0]:
%sql
-- Trata categorias, incorpora a tradução e padroniza os nomes dos atributos de produtos.

CREATE OR REPLACE TABLE workspace.silver.products
USING DELTA
AS
SELECT
    p.product_id,

    COALESCE(
        p.product_category_name,
        'sem_categoria'
    ) AS product_category_name,

    CASE
        WHEN p.product_category_name IS NULL
            THEN 'sem_categoria'
        ELSE COALESCE(
            t.product_category_name_english,
            p.product_category_name
        )
    END AS product_category_name_english,

    CAST(p.product_name_lenght AS INT) AS product_name_length,
    CAST(p.product_description_lenght AS INT) AS product_description_length,
    CAST(p.product_photos_qty AS INT) AS product_photos_qty,
    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm

FROM workspace.bronze.products p

LEFT JOIN workspace.bronze.category_translation t
    ON p.product_category_name = t.product_category_name;

num_affected_rows,num_inserted_rows


### 04.3 Validação

Após a persistência, são realizadas validações para verificar a manutenção da granularidade, a quantidade de registros e a aplicação correta das transformações definidas.

In [0]:
%sql
-- Valida a quantidade de produtos e a unicidade de product_id após o tratamento.

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT product_id) AS qtd_produtos,
    COUNT(*) - COUNT(DISTINCT product_id) AS qtd_duplicados
FROM workspace.silver.products;

qtd_registros,qtd_produtos,qtd_duplicados
32951,32951,0


In [0]:
%sql
-- Valida se categorias e traduções ficaram preenchidas após as regras aplicadas.

SELECT
    SUM(CASE WHEN product_category_name IS NULL THEN 1 ELSE 0 END) AS categoria_nula,
    SUM(CASE WHEN product_category_name_english IS NULL THEN 1 ELSE 0 END) AS traducao_nula,
    SUM(CASE WHEN product_category_name = 'sem_categoria' THEN 1 ELSE 0 END) AS sem_categoria
FROM workspace.silver.products;

categoria_nula,traducao_nula,sem_categoria
0,0,610


In [0]:
%sql
-- Valida os tipos e os nomes padronizados dos campos da tabela Silver.

DESCRIBE workspace.silver.products;

col_name,data_type,comment
product_id,string,Identificador único do produto. Domínio: identificador alfanumérico único e não nulo. Foram observados 32.951 produtos distintos.
product_category_name,string,null
product_category_name_english,string,null
product_name_length,int,null
product_description_length,int,null
product_photos_qty,int,null
product_weight_g,double,Peso do produto em gramas. Domínio observado: 0 a 40.425 gramas. Pode apresentar valores nulos.
product_length_cm,double,Comprimento do produto em centímetros. Domínio observado: 7 a 105 cm. Pode apresentar valores nulos.
product_height_cm,double,Altura do produto em centímetros. Domínio observado: 2 a 105 cm. Pode apresentar valores nulos.
product_width_cm,double,Largura do produto em centímetros. Domínio observado: 6 a 118 cm. Pode apresentar valores nulos.


## 05. Tratamento de Clientes

### 05.1 Análise de Qualidade
A tabela `bronze.customers` contém os identificadores dos clientes e as informações geográficas associadas a cada registro de pedido.

Nesta etapa são avaliadas a consistência dos campos geográficos e a adequação dos tipos de dados, preservando a distinção entre `customer_id`, identificador associado ao registro do pedido, e `customer_unique_id`, utilizado para reconhecer um mesmo consumidor em diferentes pedidos.

In [0]:
%sql
-- Verifica a consistência das UFs e identifica valores fora do padrão de duas letras maiúsculas.

SELECT
    customer_state,
    COUNT(*) AS qtd_registros
FROM workspace.bronze.customers
WHERE customer_state NOT RLIKE '^[A-Z]{2}$'
GROUP BY customer_state
ORDER BY qtd_registros DESC;

customer_state,qtd_registros


In [0]:
%sql
-- Verifica registros de cidade que necessitam padronização de espaços ou caixa.

SELECT
    COUNT(*) AS qtd_registros_para_padronizar
FROM workspace.bronze.customers
WHERE customer_city <> LOWER(TRIM(customer_city));

qtd_registros_para_padronizar
0


In [0]:
%sql
-- Verifica prefixos de CEP fora do intervalo esperado de cinco dígitos.

SELECT
    COUNT(*) AS qtd_ceps_invalidos
FROM workspace.bronze.customers
WHERE customer_zip_code_prefix < 1000
   OR customer_zip_code_prefix > 99999;

qtd_ceps_invalidos
0


### 05.2 Regras e Transformações

As validações realizadas não identificaram inconsistências nos campos geográficos da tabela de clientes. As UFs seguem o padrão de duas letras maiúsculas, os nomes das cidades não apresentam necessidade de padronização de caixa ou espaços e os prefixos de CEP encontram-se dentro do intervalo esperado.

Também não foram identificados valores nulos na análise realizada anteriormente sobre a fonte.

Dessa forma, não foram necessárias transformações de conteúdo ou tipo de dado nesta tabela. Os dados serão persistidos na camada Silver mantendo a estrutura e os valores provenientes da camada Bronze.

Foram definidas as seguintes regras:

- preservação dos identificadores `customer_id` e `customer_unique_id`;
- preservação das informações geográficas sem alteração;
- manutenção da granularidade de uma linha por `customer_id`;
- persistência dos dados validados na camada Silver sem transformação de conteúdo.

In [0]:
%sql
-- Persiste os dados de clientes validados na camada Silver sem alteração de conteúdo.

CREATE OR REPLACE TABLE workspace.silver.customers
USING DELTA
AS
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
FROM workspace.bronze.customers;

num_affected_rows,num_inserted_rows


### 05.3 Validação

Após a persistência, são realizadas validações para verificar a manutenção da granularidade, a quantidade de registros e a aplicação correta das transformações definidas.

In [0]:
%sql
-- Valida o volume, a unicidade de customer_id e a preservação dos clientes únicos.

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT customer_id) AS qtd_customer_id,
    COUNT(*) - COUNT(DISTINCT customer_id) AS qtd_duplicados,
    COUNT(DISTINCT customer_unique_id) AS qtd_clientes_unicos
FROM workspace.silver.customers;

qtd_registros,qtd_customer_id,qtd_duplicados,qtd_clientes_unicos
99441,99441,0,96096


## 06. Tratamento de Avaliações

### 06.1 Análise de Qualidade
A tabela `bronze.order_reviews` contém as avaliações realizadas sobre os pedidos, incluindo nota, comentários e datas relacionadas ao processo de avaliação.

Nesta etapa são avaliadas a validade das notas, a tipagem dos campos temporais e a ocorrência de valores ausentes. A existência de múltiplas avaliações para um mesmo pedido é preservada na camada Silver, mantendo a granularidade da fonte.

In [0]:
%sql
-- Valida se os campos temporais das avaliações podem ser convertidos para TIMESTAMP.

SELECT
    SUM(
        CASE
            WHEN review_creation_date IS NOT NULL
             AND TRY_CAST(review_creation_date AS TIMESTAMP) IS NULL
            THEN 1 ELSE 0
        END
    ) AS criacao_invalida,

    SUM(
        CASE
            WHEN review_answer_timestamp IS NOT NULL
             AND TRY_CAST(review_answer_timestamp AS TIMESTAMP) IS NULL
            THEN 1 ELSE 0
        END
    ) AS resposta_invalida

FROM workspace.bronze.order_reviews;

criacao_invalida,resposta_invalida
0,0


In [0]:
%sql
-- Verifica avaliações com notas fora do domínio esperado de 1 a 5.

SELECT
    COUNT(*) AS qtd_notas_invalidas
FROM workspace.bronze.order_reviews
WHERE review_score IS NULL
   OR review_score NOT BETWEEN 1 AND 5;

qtd_notas_invalidas
0


In [0]:
%sql
-- Verifica avaliações cuja resposta ocorreu antes da data de criação.

SELECT
    COUNT(*) AS qtd_datas_inconsistentes
FROM workspace.bronze.order_reviews
WHERE review_creation_date IS NOT NULL
  AND review_answer_timestamp IS NOT NULL
  AND TRY_CAST(review_answer_timestamp AS TIMESTAMP)
      < TRY_CAST(review_creation_date AS TIMESTAMP);

qtd_datas_inconsistentes
0


### 06.2 Regras e Transformações

A análise não identificou valores inválidos nos campos temporais nem notas fora do domínio esperado de 1 a 5. Também não foram encontradas avaliações cuja data de resposta seja anterior à data de criação.

Os valores nulos existentes nos campos de título e mensagem da avaliação serão preservados, pois representam campos opcionais e sua ausência não caracteriza, necessariamente, um problema de qualidade.

A existência de múltiplas avaliações associadas a um mesmo pedido também será preservada na camada Silver. A consolidação dessas avaliações será realizada posteriormente na camada Gold, de acordo com a granularidade definida para a tabela `ft_pedidos`.

Foram definidas as seguintes regras:

- conversão de `review_creation_date` e `review_answer_timestamp` de `STRING` para `TIMESTAMP`;
- preservação dos valores nulos nos campos opcionais de comentários;
- preservação de todas as avaliações existentes na fonte;
- manutenção da granularidade de uma linha por combinação de `order_id` e `review_id`;
- manutenção dos demais campos sem alteração de conteúdo.

In [0]:
%sql
-- Converte os campos temporais das avaliações e preserva a granularidade da fonte.

CREATE OR REPLACE TABLE workspace.silver.order_reviews
USING DELTA
AS
SELECT
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    CAST(review_creation_date AS TIMESTAMP) AS review_creation_date,
    CAST(review_answer_timestamp AS TIMESTAMP) AS review_answer_timestamp
FROM workspace.bronze.order_reviews;

num_affected_rows,num_inserted_rows


### 06.3 Validação

Após a persistência, são realizadas validações para verificar a manutenção da granularidade, a quantidade de registros e a aplicação correta das transformações definidas.

In [0]:
%sql
-- Valida o volume e a unicidade da combinação order_id + review_id após o tratamento.

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT CONCAT(order_id, '|', review_id)) AS qtd_avaliacoes_unicas,
    COUNT(*) - COUNT(DISTINCT CONCAT(order_id, '|', review_id)) AS qtd_duplicados
FROM workspace.silver.order_reviews;

qtd_registros,qtd_avaliacoes_unicas,qtd_duplicados
99224,99224,0


In [0]:
%sql
-- Confere a preservação dos valores nulos nos campos opcionais de comentários.

SELECT
    SUM(CASE WHEN review_comment_title IS NULL THEN 1 ELSE 0 END) AS titulos_nulos,
    SUM(CASE WHEN review_comment_message IS NULL THEN 1 ELSE 0 END) AS mensagens_nulas
FROM workspace.silver.order_reviews;

titulos_nulos,mensagens_nulas
87656,58247


In [0]:
%sql
-- Valida os tipos de dados resultantes após a conversão dos campos temporais.

DESCRIBE workspace.silver.order_reviews;

col_name,data_type,comment
review_id,string,Identificador da avaliação. Domínio: identificador alfanumérico não nulo. Foram observados 98.410 review_id distintos. O campo isoladamente não identifica unicamente um registro da tabela.
order_id,string,Identificador do pedido ao qual a avaliação está associada. Domínio: identificador alfanumérico não nulo. Foram observados 98.673 pedidos distintos.
review_score,bigint,"Nota atribuída na avaliação do pedido. Domínio observado: valores inteiros de 1 a 5, sem valores nulos."
review_comment_title,string,Título do comentário informado na avaliação. Domínio: texto livre e opcional. Pode apresentar valores nulos.
review_comment_message,string,Mensagem textual informada na avaliação. Domínio: texto livre e opcional. Pode apresentar valores nulos.
review_creation_date,timestamp,null
review_answer_timestamp,timestamp,null


## 07. Tratamento de Pagamentos

### 07.1 Análise de Qualidade
A tabela `bronze.order_payments` contém os registros de pagamento associados aos pedidos, podendo existir mais de um registro e mais de uma forma de pagamento para um mesmo pedido.

Nesta etapa são avaliados a consistência dos tipos de pagamento, os valores de parcelamento e os valores monetários. A granularidade original de uma linha por combinação de pedido e sequência de pagamento será preservada na camada Silver.

In [0]:
%sql
-- Quantifica registros com parcelamento igual a zero e identifica os respectivos tipos de pagamento.

SELECT
    payment_type,
    COUNT(*) AS qtd_registros
FROM workspace.bronze.order_payments
WHERE payment_installments = 0
GROUP BY payment_type
ORDER BY qtd_registros DESC;

payment_type,qtd_registros
credit_card,2


In [0]:
%sql
-- Analisa os registros cujo tipo de pagamento não foi definido na fonte.

SELECT
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
FROM workspace.bronze.order_payments
WHERE payment_type = 'not_defined';

order_id,payment_sequential,payment_type,payment_installments,payment_value
c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0


In [0]:
%sql
-- Analisa os registros com valor de pagamento igual a zero.

SELECT
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
FROM workspace.bronze.order_payments
WHERE payment_value = 0
ORDER BY payment_type, order_id;

order_id,payment_sequential,payment_type,payment_installments,payment_value
00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0
fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0


### 07.2 Regras e Transformações

A análise identificou dois registros de pagamento com `payment_installments` igual a zero, ambos associados ao tipo `credit_card`. Também foram encontrados três registros com `payment_type` igual a `not_defined` e nove registros com `payment_value` igual a zero, sendo três deles os próprios pagamentos sem tipo definido e os demais associados a vouchers.

Como não existem informações suficientes na fonte para determinar valores alternativos para esses registros, eles serão preservados na camada Silver, evitando correções ou imputações sem evidência.

Também foi identificada a necessidade de adequação dos tipos de dados. O campo `payment_value`, originalmente armazenado como `DOUBLE`, será convertido para `DECIMAL(10,2)`, garantindo precisão adequada para valores monetários. Os campos `payment_sequential` e `payment_installments`, que representam valores inteiros de pequena magnitude, serão convertidos de `BIGINT` para `INT`.

Foram definidas as seguintes regras:

- conversão de `payment_value` de `DOUBLE` para `DECIMAL(10,2)`;
- conversão de `payment_sequential` e `payment_installments` de `BIGINT` para `INT`;
- preservação dos registros com `payment_installments` igual a zero;
- preservação do tipo de pagamento `not_defined`;
- preservação dos registros com `payment_value` igual a zero;
- manutenção da granularidade de uma linha por combinação de `order_id` e `payment_sequential`;
- manutenção dos demais campos sem alteração de conteúdo.

In [0]:
%sql
-- Adequa os tipos dos campos numéricos e monetários preservando os valores da fonte.

CREATE OR REPLACE TABLE workspace.silver.order_payments
USING DELTA
AS
SELECT
    order_id,
    CAST(payment_sequential AS INT) AS payment_sequential,
    payment_type,
    CAST(payment_installments AS INT) AS payment_installments,
    CAST(payment_value AS DECIMAL(10,2)) AS payment_value
FROM workspace.bronze.order_payments;

num_affected_rows,num_inserted_rows


### 07.3 Validação

In [0]:
%sql
-- Valida o volume e a unicidade da combinação order_id + payment_sequential.

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT CONCAT(order_id, '|', payment_sequential)) AS qtd_pagamentos_unicos,
    COUNT(*) - COUNT(DISTINCT CONCAT(order_id, '|', payment_sequential)) AS qtd_duplicados
FROM workspace.silver.order_payments;

qtd_registros,qtd_pagamentos_unicos,qtd_duplicados
103886,103886,0


In [0]:
%sql
-- Valida a preservação dos casos atípicos e dos valores monetários após as transformações.

SELECT
    SUM(CASE WHEN s.payment_installments = 0 THEN 1 ELSE 0 END) AS parcelas_zero,
    SUM(CASE WHEN s.payment_type = 'not_defined' THEN 1 ELSE 0 END) AS tipo_nao_definido,
    SUM(CASE WHEN s.payment_value = 0 THEN 1 ELSE 0 END) AS valor_zero,
    SUM(
        CASE
            WHEN CAST(b.payment_value AS DECIMAL(10,2)) <> s.payment_value
            THEN 1 ELSE 0
        END
    ) AS divergencias_valor
FROM workspace.bronze.order_payments b
INNER JOIN workspace.silver.order_payments s
    ON b.order_id = s.order_id
   AND b.payment_sequential = s.payment_sequential;

parcelas_zero,tipo_nao_definido,valor_zero,divergencias_valor
2,3,9,0


In [0]:
%sql
-- Valida os tipos de dados resultantes após as transformações.

DESCRIBE workspace.silver.order_payments;

col_name,data_type,comment
order_id,string,Identificador do pedido ao qual o pagamento está associado. Domínio: identificador alfanumérico não nulo. Foram observados 99.440 pedidos distintos.
payment_sequential,int,null
payment_type,string,"Meio de pagamento utilizado. Domínio observado: credit_card, boleto, voucher, debit_card e not_defined. Sem valores nulos."
payment_installments,int,null
payment_value,"decimal(10,2)",null


## 08. Validação Geral da Camada Silver

Após a aplicação das regras de qualidade e transformação, foi realizada uma validação consolidada das tabelas persistidas na camada Silver.

O objetivo desta etapa é verificar a disponibilidade das tabelas tratadas e seus respectivos volumes, garantindo que as fontes necessárias para a construção do modelo analítico estejam devidamente persistidas antes do início da camada Gold.

In [0]:
%sql
-- Apresenta as tabelas Silver e seus volumes após a conclusão dos tratamentos.

SELECT 'orders' AS tabela, COUNT(*) AS qtd_registros
FROM workspace.silver.orders

UNION ALL

SELECT 'order_items', COUNT(*)
FROM workspace.silver.order_items

UNION ALL

SELECT 'products', COUNT(*)
FROM workspace.silver.products

UNION ALL

SELECT 'customers', COUNT(*)
FROM workspace.silver.customers

UNION ALL

SELECT 'order_reviews', COUNT(*)
FROM workspace.silver.order_reviews

UNION ALL

SELECT 'order_payments', COUNT(*)
FROM workspace.silver.order_payments

ORDER BY tabela;

tabela,qtd_registros
customers,99441
order_items,112650
order_payments,103886
order_reviews,99224
orders,99441
products,32951


> **Observação:** nem todas as tabelas da camada Bronze originaram uma tabela independente na camada Silver. A tabela de tradução de categorias foi incorporada ao tratamento de `products`, enquanto os atributos da tabela `sellers` não são necessários para o modelo analítico definido neste MVP. O identificador do vendedor utilizado nas análises é preservado diretamente em `order_items`.

## 09. Catalogo de Dados da Camada Silver


In [0]:
%sql
-- Exibe a estrutura, os tipos e os comentários atuais das tabelas da camada Silver.
DESCRIBE TABLE workspace.silver.orders;
DESCRIBE TABLE workspace.silver.order_items;
DESCRIBE TABLE workspace.silver.products;
DESCRIBE TABLE workspace.silver.customers;
DESCRIBE TABLE workspace.silver.order_reviews;
DESCRIBE TABLE workspace.silver.order_payments;

col_name,data_type,comment
order_id,string,Identificador do pedido ao qual o pagamento está associado. Domínio: identificador alfanumérico não nulo. Foram observados 99.440 pedidos distintos.
payment_sequential,int,null
payment_type,string,"Meio de pagamento utilizado. Domínio observado: credit_card, boleto, voucher, debit_card e not_defined. Sem valores nulos."
payment_installments,int,null
payment_value,"decimal(10,2)",null


In [0]:
%sql
-- Documenta o contexto da tabela Silver de pedidos.
COMMENT ON TABLE workspace.silver.orders IS
'Tabela de pedidos após aplicação das regras de qualidade e tipagem da camada Silver. Granularidade: uma linha por pedido. Os campos temporais provenientes da fonte foram convertidos de STRING para TIMESTAMP.';

-- Documenta os campos temporais tratados na tabela Silver de pedidos.
ALTER TABLE workspace.silver.orders ALTER COLUMN order_purchase_timestamp COMMENT
'Data e hora de realização do pedido. Domínio: valor temporal no formato TIMESTAMP, sem valores nulos. Convertido de STRING para TIMESTAMP na camada Silver.';

ALTER TABLE workspace.silver.orders ALTER COLUMN order_approved_at COMMENT
'Data e hora de aprovação do pedido. Domínio: valor temporal no formato TIMESTAMP. Pode apresentar valores nulos. Convertido de STRING para TIMESTAMP na camada Silver.';

ALTER TABLE workspace.silver.orders ALTER COLUMN order_delivered_carrier_date COMMENT
'Data e hora em que o pedido foi encaminhado à transportadora. Domínio: valor temporal no formato TIMESTAMP. Pode apresentar valores nulos conforme o status do pedido. Convertido de STRING para TIMESTAMP na camada Silver.';

ALTER TABLE workspace.silver.orders ALTER COLUMN order_delivered_customer_date COMMENT
'Data e hora de entrega do pedido ao cliente. Domínio: valor temporal no formato TIMESTAMP. Pode apresentar valores nulos conforme o status do pedido. Convertido de STRING para TIMESTAMP na camada Silver.';

ALTER TABLE workspace.silver.orders ALTER COLUMN order_estimated_delivery_date COMMENT
'Data e hora estimada para entrega do pedido. Domínio: valor temporal no formato TIMESTAMP, sem valores nulos. Convertido de STRING para TIMESTAMP na camada Silver.';


-- Documenta o contexto da tabela Silver de itens dos pedidos.
COMMENT ON TABLE workspace.silver.order_items IS
'Tabela de itens dos pedidos após aplicação das regras de qualidade e tipagem da camada Silver. Granularidade: uma linha por combinação de order_id e order_item_id. Campos de data e valores monetários foram adequados aos tipos utilizados no processamento analítico.';

-- Documenta a data limite de envio tratada na tabela Silver de itens.
ALTER TABLE workspace.silver.order_items ALTER COLUMN shipping_limit_date COMMENT
'Data e hora limite para envio do item pelo vendedor. Domínio observado: de 2016-09-19 a 2020-04-09, sem valores nulos. Convertido de STRING para TIMESTAMP na camada Silver.';

-- Documenta o valor do item tratado na tabela Silver de itens.
ALTER TABLE workspace.silver.order_items ALTER COLUMN price COMMENT
'Valor do produto associado ao item do pedido, em reais. Domínio observado: R$ 0,85 a R$ 6.735,00, sem valores nulos. Convertido para DECIMAL(10,2) na camada Silver.';

-- Documenta o valor do frete tratado na tabela Silver de itens.
ALTER TABLE workspace.silver.order_items ALTER COLUMN freight_value COMMENT
'Valor do frete associado ao item do pedido, em reais. Domínio observado: R$ 0,00 a R$ 409,68, sem valores nulos. Convertido para DECIMAL(10,2) na camada Silver.';


-- Documenta o contexto da tabela Silver de produtos.
COMMENT ON TABLE workspace.silver.products IS
'Tabela de produtos após aplicação das regras de qualidade, padronização e enriquecimento da camada Silver. Inclui tratamento de categorias, incorporação da tradução para inglês, correção da nomenclatura de campos e adequação de tipos. Granularidade: uma linha por produto.';

-- Documenta a categoria tratada dos produtos.
ALTER TABLE workspace.silver.products ALTER COLUMN product_category_name COMMENT
'Categoria do produto em português. Domínio original: 73 categorias distintas. Valores nulos identificados na fonte foram classificados como sem_categoria na camada Silver.';

-- Documenta a categoria em inglês incorporada na Silver.
ALTER TABLE workspace.silver.products ALTER COLUMN product_category_name_english COMMENT
'Categoria do produto em inglês, obtida a partir da tabela de tradução de categorias. Quando não existe tradução disponível, é preservado o nome original da categoria em português.';

-- Documenta o tamanho do nome após correção de nomenclatura e tipo.
ALTER TABLE workspace.silver.products ALTER COLUMN product_name_length COMMENT
'Quantidade de caracteres do nome do produto. Domínio observado: 5 a 76. Pode apresentar valores nulos. Campo renomeado a partir de product_name_lenght e convertido de DOUBLE para INT na camada Silver.';

-- Documenta o tamanho da descrição após correção de nomenclatura e tipo.
ALTER TABLE workspace.silver.products ALTER COLUMN product_description_length COMMENT
'Quantidade de caracteres da descrição do produto. Domínio observado: 4 a 3.992. Pode apresentar valores nulos. Campo renomeado a partir de product_description_lenght e convertido de DOUBLE para INT na camada Silver.';

-- Documenta a quantidade de fotos após adequação do tipo.
ALTER TABLE workspace.silver.products ALTER COLUMN product_photos_qty COMMENT
'Quantidade de fotos associadas ao produto. Domínio observado: 1 a 20. Pode apresentar valores nulos. Convertido de DOUBLE para INT na camada Silver.';


-- Documenta o contexto da tabela Silver de clientes.
COMMENT ON TABLE workspace.silver.customers IS
'Tabela de clientes utilizada na camada Silver. Granularidade: uma linha por customer_id associado ao pedido. A estrutura e os valores da fonte foram preservados por não terem sido identificadas necessidades adicionais de tratamento para os campos utilizados no modelo analítico.';


-- Documenta o contexto da tabela Silver de avaliações.
COMMENT ON TABLE workspace.silver.order_reviews IS
'Tabela de avaliações dos pedidos após aplicação das regras de qualidade e tipagem da camada Silver. Granularidade preservada por registro de avaliação associado ao pedido. Os campos temporais provenientes da fonte foram convertidos para TIMESTAMP.';

-- Documenta a data de criação da avaliação após adequação do tipo.
ALTER TABLE workspace.silver.order_reviews ALTER COLUMN review_creation_date COMMENT
'Data e hora de criação da avaliação. Domínio: valor temporal no formato TIMESTAMP. Convertido de STRING para TIMESTAMP na camada Silver.';

-- Documenta a data de resposta da avaliação após adequação do tipo.
ALTER TABLE workspace.silver.order_reviews ALTER COLUMN review_answer_timestamp COMMENT
'Data e hora de resposta da avaliação. Domínio: valor temporal no formato TIMESTAMP. Convertido de STRING para TIMESTAMP na camada Silver.';


-- Documenta o contexto da tabela Silver de pagamentos.
COMMENT ON TABLE workspace.silver.order_payments IS
'Tabela de registros de pagamento dos pedidos após aplicação das regras de qualidade e tipagem da camada Silver. Granularidade: uma linha por combinação de order_id e payment_sequential. Campos numéricos e monetários foram adequados aos tipos utilizados no processamento analítico.';

-- Documenta a sequência de pagamento após adequação do tipo.
ALTER TABLE workspace.silver.order_payments ALTER COLUMN payment_sequential COMMENT
'Número sequencial do pagamento dentro do pedido. Domínio observado: valores inteiros de 1 a 29, sem valores nulos. Em conjunto com order_id define a granularidade da tabela. Convertido para INT na camada Silver.';

-- Documenta a quantidade de parcelas após adequação do tipo.
ALTER TABLE workspace.silver.order_payments ALTER COLUMN payment_installments COMMENT
'Quantidade de parcelas associadas ao pagamento. Domínio observado: valores inteiros de 0 a 24, sem valores nulos. Os valores originais da fonte foram preservados e o campo foi convertido para INT na camada Silver.';

-- Documenta o valor do pagamento após adequação do tipo monetário.
ALTER TABLE workspace.silver.order_payments ALTER COLUMN payment_value COMMENT
'Valor associado ao registro de pagamento, em reais. Domínio observado: R$ 0,00 a R$ 13.664,08, sem valores nulos. Convertido para DECIMAL(10,2) na camada Silver.';